In [ ]:
USE_SIMULATOR = True

In [ ]:
from qiskit.circuit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from samplomatic import InjectNoise, Twirl

from qiskit_noise_learning.noise_learner import LearningOptions, NoiseLearner

In [ ]:
if USE_SIMULATOR:
    from qiskit_aer import AerSimulator
    from qiskit_ibm_runtime.fake_provider.backends.fez import FakeFez

    from qiskit_noise_learning.aer_executor import AerExecutor

    backend = FakeFez()
    executor = AerExecutor(AerSimulator(method="stabilizer"))
else:
    from qiskit_ibm_runtime import QiskitRuntimeService

    runtime = QiskitRuntimeService(name="ibm_cloud_premium")
    backend = runtime.backend("ibm_pittsburgh")
    executor = None

options = LearningOptions(depths=[2, 4, 8] if USE_SIMULATOR else [12, 64, 128], num_randomizations=2 if USE_SIMULATOR else 30)
noise_learner = NoiseLearner(backend, options=options, executor=executor)

In [ ]:
if USE_SIMULATOR:
    edges = [(a, b) for a, b in backend.coupling_map.get_edges() if a < b]
    layer_1_pairs = edges[:3]
    layer_2_pairs = edges[3:6]
else:
    layer_1_pairs = [(91, 92), (93, 94), (95, 99), (98, 111), (112, 113), (114, 115)]
    layer_2_pairs = [(91, 98), (92, 93), (94, 95), (99, 115), (111, 112), (113, 114)]

c = QuantumCircuit(backend.num_qubits)

with c.box([Twirl(), InjectNoise("hi")]):
    for pair in layer_1_pairs:
        c.cz(*pair)

with c.box([Twirl()]):
    for pair in layer_2_pairs:
        c.cz(*pair)

In [ ]:
job = noise_learner.run([c[0]])
job

In [ ]:
result = job.result()
result

In [ ]:
result.to_dict()